# **CST8508 Machine Vision - Lab 5: Cats vs Dogs Image Classification**

**Objective:** Implement an end-to-end Convolutional Neural Network (CNN) to classify images from the Cats vs. Dogs dataset. You can download the dataset from here https://www.microsoft.com/en-us/download/details.aspx?id=54765
Do not forget to change the runtime type of your notebook to GPU so you can train on a GPU.

**Lab Instructions:**

* Implement each function in a modular fashion.
* Ensure functions interact with each other seamlessly.
* Document each step with comments for clarity.
* After implementing all parts, run the entire pipeline on the Cats vs. Dogs dataset and analyze the results.


This lab will provide a comprehensive understanding of building and training a CNN for image classification, from data preprocessing to model evaluation.







In [1]:
# ── Cell 0: Dataset Download & Cleanup ─────────────────────────────────────────
#
# ── Imports ──
# os:              OS interface — file path operations
# zipfile:         Extract .zip archives (dataset distributed as compressed archive)
# urllib.request:  HTTP download (stdlib — no pip install needed)
# pathlib.Path:    OOP file paths (cleaner than os.path)
# PIL.Image:       Pillow imaging lib — verify() detects corrupt JPEGs
# Why all stdlib (except PIL)? → Minimal-dependency principle — fewer deps = fewer env issues
#   🪨 Irreducible: portability = code runs on any machine
#   📖 Source: Python standard library docs (https://docs.python.org/3/library/)

import os, zipfile, urllib.request
from pathlib import Path
from PIL import Image

# ── Dataset URL & paths ──
# What: Define download URL, local ZIP path, and extracted folder path
# Dataset: Microsoft Cats vs Dogs — 12,500 cats + 12,500 dogs, ~786 MB
# Why this dataset? → Classic binary-classification benchmark with enough images (25K) to train a CNN
#   Why so many images? → CNN has ~8.5M parameters — too few images → overfitting (memorisation)
#   Why is overfitting bad? → Model only memorises training set, fails on new images
#   🪨 Irreducible: generalisation = the ultimate goal of ML
#   📖 Source: Goodfellow et al. (2016), "Deep Learning", MIT Press, §5.2

DATASET_URL  = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
ZIP_PATH     = Path("kagglecatsanddogs_5340.zip")
DATASET_PATH = Path("PetImages")

# ── Download dataset ──
# What: Download from Microsoft server if neither ZIP nor extracted folder exists locally
# Why check before downloading? → 786 MB is large — re-downloading wastes time
# Why check two conditions? → ZIP may have been deleted but folder already extracted → no need to re-download
if not ZIP_PATH.exists() and not DATASET_PATH.exists():
    print("Downloading dataset (~786 MB) ...")
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
    print("Download complete.")
else:
    print("Zip / dataset already present, skipping download.")

# ── Extract dataset ──
# What: Unzip to current directory, creating PetImages/Cat/ and PetImages/Dog/ folders
# Folder layout: PetImages/Cat/*.jpg + PetImages/Dog/*.jpg
# Why this layout? → PyTorch ImageFolder uses folder names as class labels (Cat=0, Dog=1) [ImageFolder]
#   Why not a CSV list? → ImageFolder = zero-config, folder name IS the label
#   🪨 Irreducible: convention over configuration — reduces human error
#   📖 Source: [ImageFolder] PyTorch docs — torchvision.datasets.ImageFolder
if not DATASET_PATH.exists():
    print("Extracting ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(".")
    print("Extraction complete.")

# ── Remove corrupted images ──
# What: Check each JPEG with PIL verify(), delete any corrupted files
# Why verify? → This dataset has known broken JPEGs
#   Why do corrupted images cause problems? → DataLoader decode failure → entire training crashes
#   Why can't we try-except and skip? → DataLoader's collate_fn raises on any exception — no skip
#   Why do JPEGs get corrupted? → JPEG is compressed; even 1 wrong byte breaks the Huffman table
#   🪨 Irreducible: file integrity is a binary-level hard constraint
#   📖 Source: JPEG standard ITU-T T.81; Pillow docs — Image.verify()
removed = 0
for img_path in DATASET_PATH.rglob("*.jpg"):
    try:
        with Image.open(img_path) as img:
            img.verify()
    except Exception:
        img_path.unlink()
        removed += 1
print(f"Removed {removed} corrupted images.")
print(f"Dataset ready at: {DATASET_PATH.resolve()}")

**Part 1:** Data Loading and Augmentation

**Function load_dataset(path):** Load the Cats vs. Dogs dataset from the given path. Split into training and test sets. Define transforms for augmentation and normalization.


In [2]:
# ── Cell 1: Data Loading & Augmentation ───────────────────────────────────────
#
# ── Imports ──
# random:                     Control randomness of data split — reproducibility
# torch:                      PyTorch core (tensor ops + autograd + GPU acceleration)
# DataLoader:                 Auto batching, shuffling, multi-process prefetch
# Subset:                     Slice train/test subsets by index list (stores indices only, not images)
# datasets.ImageFolder:       Assigns class labels by folder name [ImageFolder]
# transforms:                 Image preprocessing pipeline (Resize / Flip / Normalize etc.)

import os, random
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# ── Fix random seed ──
# What: Set random and torch seeds to 42
# Why fix seeds? → Ensures train/test split is identical every run → reproducible results
#   Why reproducibility? → Without it, impossible to compare "did the parameter change help or hurt?"
#   🪨 Irreducible: controlled variables = fundamental requirement of scientific experiments
#   📖 Source: PyTorch docs — Reproducibility (https://pytorch.org/docs/stable/notes/randomness.html)
random.seed(42)
torch.manual_seed(42)

# ── Select compute device ──
# What: Detect whether GPU is available; prefer GPU
# Why detect CUDA? → GPU training is 5-10× faster (parallel matrix computation)
#   Why is GPU faster? → GPU has thousands of small cores for parallel matrix ops
#   🪨 Irreducible: parallel computing is a hardware architectural advantage
#   📖 Source: NVIDIA CUDA Programming Guide
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# What: Define data loading function — read images, augment, split train/test, wrap in DataLoader
# Why wrap in a function? → Main pipeline only needs one line: load_dataset()
#   Why parameterise split_ratio? → Easy to experiment with different splits (80/20, 70/30 etc.)
#   🪨 Irreducible: modularity = reusable + testable + maintainable
#   📖 Source: Martin (2003), "Clean Code", Prentice Hall — Single Responsibility Principle
def load_dataset(path, split_ratio=0.8):

    # ══════════════════════════════════════════════════
    # Training transforms (with augmentation)
    # What: Chain multiple image transforms into a pipeline using transforms.Compose
    #       Training set adds Flip and Rotation for data augmentation (test set does not)
    # Why augmentation? → Let model see more "variants", prevent memorising raw images [DataAug]
    #   Why augment only training set? → Test set simulates real-world — must not be artificially modified
    #   🪨 Irreducible: training = studying, testing = exam — no open-book exams
    #   📖 Source: [DataAug] Shorten & Khoshgoftaar (2019), J Big Data 6(1), §3
    # ══════════════════════════════════════════════════
    train_transform = transforms.Compose([

        # What: Resize all images to 128×128 pixels
        # Why Resize? → Original images vary in size; must unify for batching
        #   Why 128? → Smaller than ImageNet standard (224) → 3× faster training, sufficient for lab
        #   🪨 Irreducible: GPU parallel computation requires uniform tensor shapes
        #   📖 Source: PyTorch docs — DataLoader collate_fn requires same-shape tensors
        transforms.Resize((128, 128)),

        # What: Randomly flip images horizontally with 50% probability (data augmentation)
        # Why flip? → Creates new samples for free — effectively doubles dataset [DataAug]
        #   Why doesn't flipping break labels? → A flipped cat is still a cat → semantics preserved
        #   🪨 Irreducible: increase diversity while preserving labels → forces learning essence
        #   📖 Source: [DataAug] Shorten & Khoshgoftaar (2019); Krizhevsky et al. (2012), ImageNet §4.1
        transforms.RandomHorizontalFlip(),

        # What: Randomly rotate images by ±15° (data augmentation)
        # Why rotate? → Simulates camera tilt when taking photos [DataAug]
        #   Why ±15°? → Too large (90°) creates unnatural blanks; too small (2°) has no effect
        #   🪨 Irreducible: simulate real-world data distribution → improve generalisation
        #   📖 Source: [DataAug] Shorten & Khoshgoftaar (2019), §3 — Geometric Transformations
        transforms.RandomRotation(15),

        # What: Convert PIL image to PyTorch Tensor, pixel values [0,255] → [0,1]
        # Why ToTensor? → PyTorch only operates on Tensors — this step is mandatory
        #   🪨 Irreducible: framework hard requirement for input format
        #   📖 Source: PyTorch docs — torchvision.transforms.ToTensor
        transforms.ToTensor(),

        # What: Normalise each RGB channel (subtract mean, divide by std) → mean≈0, std≈1
        # Why Normalise? → Gradient descent converges fastest when dimensions at uniform scale [IN-norm]
        #   Why ImageNet values? → Statistics from 1.2M images — industry standard
        #   🪨 Irreducible: gradient descent converges fastest on a spherical surface — math property
        #   📖 Source: [ImageNet] Deng et al. (2009), CVPR; [IN-norm] PyTorch Normalize docs
        transforms.Normalize([0.485, 0.456, 0.406],   # RGB means
                             [0.229, 0.224, 0.225]),   # RGB stds
    ])

    # ══════════════════════════════════════════════════
    # Test transforms (NO augmentation — only Resize + ToTensor + Normalize)
    # Why no Flip or Rotation? → Test = simulate real-world; augmentation is training-only
    #   🪨 Irreducible: evaluation must be done on unmodified original data to be meaningful
    #   📖 Source: Goodfellow et al. (2016), §5.2 — train/test separation principle
    # ══════════════════════════════════════════════════
    test_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    # What: Read entire dataset using ImageFolder — folder names become labels automatically
    # Why ImageFolder? → Zero-config loading; folder name = label, no CSV mapping needed
    #   🪨 Irreducible: convention over configuration — reduces human error
    #   📖 Source: [ImageFolder] PyTorch docs — torchvision.datasets.ImageFolder
    full = datasets.ImageFolder(root=path, transform=train_transform)
    n = len(full)

    # What: Split into training and test sets at 80/20 ratio
    # Why 80/20? → Standard split — ~20,000 train, ~5,000 test [CLT]
    #   Why not 50/50? → Too little training data to learn sufficient patterns
    #   Why 5,000 enough? → Statistics: 5,000 samples give accuracy CI < ±1%
    #   🪨 Irreducible: evaluation must be on "in-distribution but unseen" data
    #   📖 Source: [CLT] Casella & Berger, "Statistical Inference" §5.5; Goodfellow et al. (2016) §5.2
    train_n = int(n * split_ratio)

    # What: Shuffle indices before splitting — avoids first half = all cats, second half = all dogs
    # Why shuffle? → Raw data is sorted by folder; without shuffling train = all cats
    #   🪨 Irreducible: IID (independent & identically distributed) is a prerequisite for SGD convergence
    #   📖 Source: Goodfellow et al. (2016), §8.1.3 — SGD convergence assumptions
    idx = list(range(n))
    random.shuffle(idx)
    train_data = Subset(full, idx[:train_n])
    test_data  = Subset(datasets.ImageFolder(root=path, transform=test_transform), idx[train_n:])

    # What: Create DataLoader — fetch 32 images per batch
    # Why batch_size=32? → Compromise between GPU memory and gradient estimate quality [MiniBatch]
    #   Why not all 20,000 at once? → GPU VRAM can't hold it
    #   Why not one at a time? → Single-sample gradient is extremely noisy → very slow convergence
    #   Why 32? → Central Limit Theorem: mean of 32 samples is a reasonable approximation [CLT]
    #   🪨 Irreducible: mini-batch = Monte Carlo approximation of true gradient under finite resources
    #   📖 Source: [MiniBatch] Goodfellow et al. (2016), §8.1.3; [CLT] Casella & Berger, §5.5
    #
    # shuffle=True  → without shuffling, periodic bias in updates
    # num_workers=2 → child processes prefetch next batch while GPU computes
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True,  num_workers=2)

    # shuffle=False → test does not need shuffling; deterministic order aids reproducibility
    test_loader  = DataLoader(test_data,  batch_size=32, shuffle=False, num_workers=2)
    print(f"Train: {train_n}  Test: {n - train_n}  Classes: {full.classes}")
    return train_loader, test_loader

**Part 2:** Model Definition

**Function define_model():** Define a CNN model with layers (Conv2D, MaxPooling, Flatten, Dense). Include activation functions and the optimizer.

In [3]:
# ── Cell 2: CNN Model Definition ───────────────────────────────────────────────
#
# ── Imports ──
# torch.nn:            Layer definitions (Conv2d / Linear / Dropout / MaxPool2d / Module base class)
# torch.nn.functional: Stateless functional API (relu / softmax) — no instantiation needed

import torch.nn as nn
import torch.nn.functional as F

# What: Define SimpleCNN class, inheriting nn.Module (base class for all PyTorch models)
# Why inherit Module? → Automatic parameter registration, GPU transfer, train/eval mode switching
#   Why let framework manage parameters? → Tracking 8.5M weights manually is impossible → automate
#   🪨 Irreducible: "everything is a Module" is PyTorch's core abstraction
#   📖 Source: PyTorch docs — torch.nn.Module (https://pytorch.org/docs/stable/nn.html)
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        # What: Conv layer 1 — input 3-channel RGB, output 32 feature maps, 3×3 kernel, padding=1
        #
        # Param in_channels=3
        # Why 3? → RGB colour images have red/green/blue channels
        #   🪨 Irreducible: images are physically 3-channel representations
        #   📖 Source: Gonzalez & Woods (2018), "Digital Image Processing" §6.1
        #
        # Param kernel_size=3
        # Why 3×3? → VGGNet proved: two 3×3 = one 5×5 receptive field, but 44% fewer params [VGG]
        #   Why local rather than global? → Image features are locally composed: edges→textures→parts
        #   🪨 Irreducible: images have spatial locality — nearby pixels strongly correlated [VGG §2.3]
        #   📖 Source: [VGG] Simonyan & Zisserman (2015), ICLR, §2.1–2.3 (arXiv:1409.1556)
        #
        # Param padding=1
        # Why pad with zeros? → Keeps spatial size unchanged: (128+2×1−3)/1+1 = 128
        #   Why preserve size? → Downsampling controlled entirely by MaxPool → cleaner design
        #   🪨 Irreducible: make spatial compression controllable — one mechanism (Pool) manages it
        #   📖 Source: [VGG] Simonyan & Zisserman (2015), §2.1 — same-padding + pooling design
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)

        # What: Conv layers 2 & 3 — channel count doubles each layer: 32→64→128
        # Why increase? → Shallow layers learn simple features (edges); deep layers learn complex ones
        #   Why double? → Each Pool halves spatial dims (area ÷ 4); doubling channels compensates
        #   🪨 Irreducible: CNN core = layer-by-layer from "high spatial + simple" to "low spatial + rich"
        #   📖 Source: Zeiler & Fergus (2014), "Visualizing and Understanding CNNs", ECCV
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        # What: 2×2 max-pooling — take max in each 2×2 window, halving width and height
        # Why max? → Maximum = strongest feature response in that region [MaxPool]
        #   Why reduce? → Without reduction FC input = 128×128×128 = 2M dims → OOM + overfitting
        #   Why exactly 3 pools? → 128 ÷ 2 ÷ 2 ÷ 2 = 16; one more → 8, too small
        #   🪨 Irreducible: pooling = lossy compression — keep most salient features, discard redundancy
        #   📖 Source: [MaxPool] Scherer et al. (2010), "Evaluation of Pooling Operations", ICANN, §3
        self.pool = nn.MaxPool2d(2, 2)

        # What: FC layer 1 — compress 128×16×16 = 32,768 dims to 256
        # Why 128×16×16? → After 3 pools: 128 channels × (128÷2÷2÷2=16) × 16
        # Why 256? → Dimensionality reduction for classification
        #   🪨 Irreducible: dimensionality = trade-off between representational power and overfitting
        #   📖 Source: Goodfellow et al. (2016), §5.2 — bias-variance tradeoff
        self.fc1 = nn.Linear(128 * 16 * 16, 256)

        # What: Dropout — randomly disable 50% of FC1 neurons during training
        # Why Dropout? → FC1 has 32768×256 ≈ 8.39M params (99% of total) — most prone to overfitting
        #   Why 0.5? → Hinton's original paper: p=0.5 maximises sub-network combinations [Dropout §4]
        #   🪨 Irreducible: Dropout = implicit ensemble. Each step trains a different sub-network [Dropout §7]
        #   📖 Source: [Dropout] Srivastava et al. (2014), JMLR 15, §4 & §7
        self.dropout = nn.Dropout(0.5)

        # What: FC layer 2 (output) — 2 logits: [cat_score, dog_score]
        # Why 2 not 1? → CrossEntropyLoss requires multi-class format (internal Softmax needs ≥ 2)
        #   Why output logits not probabilities? → Numerical stability; framework fuses Softmax+log
        #   🪨 Irreducible: floating-point precision limits; fused computation avoids log(0)
        #   📖 Source: PyTorch docs — CrossEntropyLoss (LogSoftmax + NLLLoss fused)
        self.fc2 = nn.Linear(256, 2)

    def forward(self, x):                          # x: (B, 3, 128, 128)

        # What: Each layer: Conv (preserve size) → ReLU (non-linearity) → Pool (halve size)
        #
        # ReLU = max(0, x): negatives → 0, positives unchanged
        # Why need activation? → Without one, multiple linear layers = single linear → can only draw lines
        #   Why not Sigmoid? → Sigmoid max derivative = 0.25; 10 layers: 0.25^10 ≈ 1e-6 → vanishing gradient
        #   Why ReLU doesn't vanish? → For x > 0, derivative = 1; chained product stays 1 [ReLU]
        #   🪨 Irreducible: backprop = chain rule. derivative < 1 chained → 0 (vanish); = 1 → stable [Backprop]
        #   📖 Source: [ReLU] Nair & Hinton (2010), ICML; [Backprop] Rumelhart et al. (1986), Nature 323
        x = self.pool(F.relu(self.conv1(x)))       # → (B, 32, 64, 64)
        x = self.pool(F.relu(self.conv2(x)))       # → (B, 64, 32, 32)
        x = self.pool(F.relu(self.conv3(x)))       # → (B, 128, 16, 16)

        # What: Flatten + fully connected + Dropout + output
        x = x.view(x.size(0), -1)                 # Flatten: (B, 32768)
        x = F.relu(self.fc1(x))                    # → (B, 256)
        x = self.dropout(x)                        # drop 50% neurons during training
        x = self.fc2(x)                            # → (B, 2) logits
        return x
        # NOTE: no Softmax here — CrossEntropyLoss includes it internally.

# ══ Parameter count ══
# Conv1: 3×32×3×3 + 32         =       896
# Conv2: 32×64×3×3 + 64        =    18,496
# Conv3: 64×128×3×3 + 128      =    73,856
# FC1:   32768×256 + 256        = 8,388,864  ← 99%!
# FC2:   256×2 + 2              =       514
# Total:                         ≈ 8,483,000

def define_model():
    model = SimpleCNN().to(DEVICE)
    return model

**Part 3:** Model Training

**Function train_model(model, train_data, validation_data):** Train the model using the training set with validation data. Set epochs and batch size.


In [4]:
# ── Cell 3: Model Training ────────────────────────────────────────────────────
#
# ── Import ──
# torch.optim: Optimiser collection (SGD / Adam / AdamW gradient update algorithms)

import torch.optim as optim

# What: Set training epochs to 10 (pass through all training data 10 times)
# Why multiple passes? → One pass is not enough to learn (like reading a textbook once)
#   Why not 100? → Too many → model memorises noise (overfitting)
#   Why 10? → val_acc is still rising (88%) → still learning but enough for demo
#   🪨 Irreducible: epoch count = balance point between underfitting and overfitting
#   📖 Source: Goodfellow et al. (2016), §7.8 — Early Stopping
def train_model(model, train_loader, test_loader, epochs=10):
    # What: Cross-entropy loss = −log(predicted probability of the correct class)
    # Why cross-entropy? → Information-theoretically optimal for classification [CrossEntropy]
    #   Why not MSE? → When prediction is very wrong, MSE gradient is small; CE gradient is large
    #   Why −log? → Information theory: −log(p) = "surprise". p=0.9→loss=0.1; p=0.1→loss=2.3 [KL]
    #   🪨 Irreducible: CE derives from KL divergence — optimal measure of distribution difference
    #   📖 Source: [CrossEntropy] Goodfellow et al. (2016), §6.2.2; [KL] Shannon (1948)
    criterion = nn.CrossEntropyLoss()

    # What: Adam optimiser with lr=0.001
    # Why Adam? → Per-parameter adaptive learning rates → no manual tuning needed [Adam]
    #   Why lr=0.001? → Default from paper. Too large (0.1) → oscillation; too small → no progress
    #   🪨 Irreducible: gradient descent = finding lowest point on high-dim surface. Adam auto-tunes.
    #   📖 Source: [Adam] Kingma & Ba (2015), ICLR, §2 Algorithm 1 (arXiv:1412.6980)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, epochs + 1):

        # ════ Training phase ════
        # What: model.train() enables Dropout
        # Why call train()? → Enables Dropout (random neuron masking) [PyTorch-train/eval]
        #   🪨 Irreducible: training and inference are two different forward-pass behaviours
        #   📖 Source: [PyTorch-train/eval] PyTorch docs — Module.train() / Module.eval()
        model.train()

        train_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:            # 32 images per batch
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            # ① Zero out old gradients
            # Why? → PyTorch accumulates by default; without clearing, old gradients contaminate
            #   🪨 Irreducible: gradients must reflect current batch's partial derivatives
            #   📖 Source: PyTorch docs — Optimizer.zero_grad()
            optimizer.zero_grad()

            # ② Forward pass → ③ Compute loss
            out = model(imgs)                        # → (32, 2) logits
            loss = criterion(out, labels)

            # ④ Backward pass — chain rule computes ∂loss/∂w for every parameter
            #   🪨 Irreducible: calculus chain rule: df/dx = df/dy × dy/dx
            #   📖 Source: [Backprop] Rumelhart et al. (1986), Nature 323; Goodfellow et al. (2016), §6.5
            loss.backward()

            # ⑤ Update weights: w ← w − lr × grad
            optimizer.step()

            # Summary: zero_grad → forward → loss → backward → step

            # Accumulate batch loss and correct count
            train_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_loss /= total
        train_acc = correct / total

        # ════ Validation phase ════
        # What: model.eval() disables Dropout → deterministic output
        model.eval()

        val_loss, val_correct, val_total = 0.0, 0, 0

        # What: Disable gradient computation (saves VRAM + faster)
        # Why no_grad? → Validation doesn't need backprop; skip intermediate storage
        #   🪨 Irreducible: gradient computation requires storing intermediates; no gradients = no storage
        #   📖 Source: PyTorch docs — torch.no_grad()
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs)
                val_loss += criterion(out, labels).item() * imgs.size(0)
                val_correct += (out.argmax(1) == labels).sum().item()
                val_total += labels.size(0)
        val_loss /= val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch [{epoch:02d}/{epochs}]  train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    return history

**Part 4:** Model Evaluation

**Function evaluate_and_predict
(model, test_loader):** Evaluate the model's performance on the test dataset. Return accuracy

In [5]:
# ── Cell 4: Model Evaluation ──────────────────────────────────────────────────
#
# What: Evaluate model on test set, produce accuracy + classification report
# Why separate function? → Evaluation logic decoupled from training — can evaluate any model anytime
#   🪨 Irreducible: decoupling training and evaluation = more flexible experiment workflow
#   📖 Source: Software engineering — Separation of Concerns principle

def evaluate_and_predict(model, test_loader):
    # What: Switch to evaluation mode (disable Dropout)
    # Why eval()? → Training uses Dropout; evaluation needs all neurons for deterministic output
    #   🪨 Irreducible: evaluation must be deterministic — same input → same output
    #   📖 Source: [PyTorch-train/eval] PyTorch docs — Module.train() / Module.eval()
    model.eval()
    predictions, actual_labels = [], []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE)

            # What: Pick class with highest logit (0=Cat, 1=Dog)
            # Why argmax? → Larger logit = higher confidence for that class
            #   🪨 Irreducible: classification = pick the class with highest probability
            #   📖 Source: Bishop (2006), "Pattern Recognition and ML", §1.5.4
            preds = model(imgs).argmax(1)

            # What: Move GPU tensor to CPU and convert to Python list
            # Why .cpu()? → sklearn is CPU-only; .tolist() → sklearn needs Python lists
            #   🪨 Irreducible: data format compatibility between libraries is mandatory
            #   📖 Source: scikit-learn docs — sklearn.metrics
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.tolist())

    accuracy = sum(p == a for p, a in zip(predictions, actual_labels)) / len(actual_labels)
    print(f"Accuracy: {accuracy:.4f}")

    # What: Print classification report (per-class precision / recall / F1)
    # Why not just accuracy? → Accuracy hides per-class performance differences
    #   🪨 Irreducible: a single metric hides class-level performance gaps
    #   📖 Source: Sokolova & Lapalme (2009), "A systematic analysis of performance measures"
    # Precision = TP/(TP+FP): "of those predicted Cat, how many are really Cat?"
    # Recall    = TP/(TP+FN): "of all real cats, how many were found?"
    # F1        = harmonic mean of Precision & Recall
    from sklearn.metrics import classification_report
    print(classification_report(actual_labels, predictions, target_names=['Cat', 'Dog']))

    return accuracy, predictions, actual_labels

**Running the code**

In [6]:
# ── Cell 5: Main Pipeline ─────────────────────────────────────────────────────
#
# What: 4-step pipeline — load data → create model → train → evaluate
# Modular design: swap model by changing SimpleCNN(), swap data by changing the path
train_loader, test_loader = load_dataset(str(DATASET_PATH))
model = define_model()
train_model(model, train_loader, test_loader, epochs=10)
accuracy, predictions, actual_labels = evaluate_and_predict(model, test_loader)